# Case Prioritization Predictive

## Problem Framing

**Business question:** Which residents most urgently need staff follow-up in the next 60 days?

**Operational decision supported:** Help staff triage limited time toward residents with the strongest short-term follow-up signal.

**Primary users:** case managers, safehouse leadership

**Target summary:** Current predictive label: `label_case_prioritization_next_60d`, combining future incidents with coordinated counseling and visitation follow-up signals.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='case_prioritization',
    dataset_name='resident_monthly_features',
    predictive_impl='case_prioritization',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


,resident_id,snapshot_month,safehouse_id,case_status,case_category,sex,initial_risk_level,months_since_admission,age_years_at_snapshot,subcategory_flag_count,...,future_window_complete_60d,future_window_complete_90d,future_window_complete_120d,label_incident_next_30d,label_case_prioritization_next_60d,label_counseling_progress_next_90d,label_education_improvement_next_120d,label_wellbeing_deterioration_next_90d,label_supportive_home_visit_next_120d,label_reintegration_complete_next_90d
0,50,2023-01-01,4,Active,Neglected,F,High,0,11.90,2,...,True,True,True,False,0,False,True,False,False,False
1,23,2023-02-01,5,Closed,Foundling,F,High,0,9.07,1,...,True,True,True,False,1,True,True,False,False,False
2,45,2023-02-01,3,Transferred,Abandoned,F,Medium,0,14.99,2,...,True,True,True,False,0,False,True,False,True,False
3,50,2023-02-01,4,Active,Neglected,F,High,1,11.98,2,...,True,True,True,False,0,False,True,False,False,False
4,13,2023-02-01,2,Closed,Surrendered,F,Medium,0,8.73,1,...,True,True,True,False,1,True,True,False,False,False
5,50,2023-03-01,4,Active,Neglected,F,High,2,12.06,2,...,True,True,True,False,0,True,True,False,False,False
6,2,2023-03-01,3,Closed,Surrendered,F,Medium,0,14.85,2,...,True,True,True,False,0,False,True,False,True,False
7,23,2023-03-01,5,Closed,Foundling,F,High,1,9.15,1,...,True,True,True,False,1,True,True,False,False,False
8,29,2023-03-01,8,Transferred,Abandoned,F,Medium,0,12.08,1,...,True,True,True,False,1,True,True,False,False,False
9,13,2023-03-01,2,Closed,Surrendered,F,Medium,1,8.80,1,...,True,True,True,False,1,True,True,False,False,False


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_case_prioritization_next_60d`, combining future incidents with coordinated counseling and visitation follow-up signals.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [3]:
evaluation = load_evaluation_bundle('case_prioritization')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'random_forest_classifier',
 'train_rows': 756,
 'test_rows': 715,
 'target_col': 'label_case_prioritization_next_60d',
 'split_col': 'snapshot_month',
 'selection_metric': 'average_precision',
 'cutoff_date': '2025-04-01',
 'task_type': 'classification',
 'sample_count': 715.0,
 'positive_count': 213.0,
 'positive_rate': 0.29790209790209793,
 'accuracy': 0.42657342657342656,
 'balanced_accuracy': 0.540280193778875,
 'precision': 0.31992687385740404,
 'recall': 0.8215962441314554,
 'f1': 0.4605263157894737,
 'roc_auc': 0.6099592241363185,
 'average_precision': 0.429288937311923}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,case_prioritization,random_forest_classifier,715.0,213.0,0.297902,0.426573,0.540280,0.319927,0.821596,0.460526,0.609959,0.429289,NaN,NaN,NaN
1,case_prioritization,logistic_regression,715.0,213.0,0.297902,0.468531,0.501239,0.298795,0.582160,0.394904,0.525036,0.314166,NaN,NaN,NaN
2,case_prioritization,dummy_classifier,715.0,213.0,0.297902,0.297902,0.500000,0.297902,1.000000,0.459052,0.500000,0.297902,NaN,NaN,NaN


In [5]:
summarize_frame(cv_summary)


,pipeline_name,model_name,fold_mean,fold_std,sample_count_mean,sample_count_std,positive_count_mean,positive_count_std,positive_rate_mean,positive_rate_std,...,roc_auc_std,average_precision_mean,average_precision_std,n_splits,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,case_prioritization,random_forest_classifier,3.0,1.581139,151.2,0.447214,85.0,0.0,0.562173,0.001656,...,0.062468,0.721534,0.055566,5,NaN,NaN,NaN,NaN,NaN,NaN
1,case_prioritization,logistic_regression,3.0,1.581139,151.2,0.447214,85.0,0.0,0.562173,0.001656,...,0.019663,0.661644,0.030481,5,NaN,NaN,NaN,NaN,NaN,NaN
2,case_prioritization,dummy_classifier,3.0,1.581139,151.2,0.447214,85.0,0.0,0.562173,0.001656,...,0.000000,0.562173,0.001656,5,NaN,NaN,NaN,NaN,NaN,NaN


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to case managers, safehouse leadership?


## Deployment Notes

Recommended shared widgets:

* `ranked_table_widget`
* `recommendation_panel`
* `insight_summary_card`

Deployment checklist:

* Use ranked caseload tables during shift handoff and case conference preparation.
* Pair the score with a short recommendation panel describing the strongest urgency drivers.

Standard endpoint pattern:

* `POST /ml/predict/case_prioritization`
* `POST /ml/score-batch/case_prioritization`
